# 🚀 Interactive Customer Churn Dashboard
**Future Interns — Task 2 | Customer Retention & Churn Analysis**

---

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('../dataset/telecom_churn_data.csv')
df['JoinDate'] = pd.to_datetime(df['JoinDate'])
df['Churn_Label'] = df['Churn'].map({1: 'Churned', 0: 'Retained'})
df['Tenure_Group'] = pd.cut(df['Tenure_Months'], bins=[0,6,12,24,36,72],
    labels=['0-6 mo','7-12 mo','13-24 mo','25-36 mo','36+ mo'])

COLORS = {'Retained': '#27ae60', 'Churned': '#c0392b'}
print('✅ Data loaded:', df.shape)

✅ Data loaded: (1500, 20)


## 📊 KPI Cards

In [2]:
total=len(df)
churned=df['Churn'].sum()
retained=total-churned
churn_rate=churned/total*100
avg_tenure=df['Tenure_Months'].mean()
avg_monthly=df['MonthlyCharges'].mean()

fig = make_subplots(
    rows=1,
    cols=6,
    specs=[[{'type':'indicator'}]*6]
)

kpis = [
    ('👥 Total Customers', total),
    ('✅ Retained', retained),
    ('❌ Churned', churned),
    ('📉 Churn Rate %', churn_rate),
    ('⏱️ Avg Tenure', avg_tenure),
    ('💰 Avg Charge (₹)', avg_monthly)
]

for i, (title, val) in enumerate(kpis, 1):
    fig.add_trace(
        go.Indicator(
            mode='number',
            value=val,
            title={
                'text': f'<b>{title}</b>',
                'font': {'size': 13, 'color': 'white'}
            },
            number={
                'font': {'size': 28, 'color': 'white'},
                'valueformat': '.1f' if isinstance(val, float) else ','
            }
        ),
        row=1,
        col=i
    )

fig.update_layout(
    height=180,
    title_text='📊 Key Performance Indicators',
    title_font_size=18,
    title_font_color='white',

    # Dark Theme
    paper_bgcolor='#1E293B',
    plot_bgcolor='#1E293B',

    font=dict(color='white'),

    margin=dict(
        t=60,
        b=10,
        l=10,
        r=10
    )
)

fig.show()

## 🍩 Churn Overview

In [3]:
fig = make_subplots(
    rows=1,
    cols=2,
    specs=[[{'type':'pie'}, {'type':'bar'}]],
    subplot_titles=['🍩 Churn vs Retention', '📊 Churn Rate by Contract']
)

cc = df['Churn_Label'].value_counts()

fig.add_trace(
    go.Pie(
        labels=cc.index,
        values=cc.values,
        hole=0.5,
        marker_colors=["#46ba39", '#c0392b'],
        textinfo='label+percent',
        textfont_size=13,
        pull=[0, 0.06]
    ),
    row=1,
    col=1
)

ct = df.groupby('Contract')['Churn'].mean().mul(100).round(1).reset_index()
ct.columns = ['Contract', 'ChurnRate']
ct = ct.sort_values('ChurnRate', ascending=False)

fig.add_trace(
    go.Bar(
        x=ct['Contract'],
        y=ct['ChurnRate'],
        marker_color=['#c0392b', '#e67e22', '#27ae60'],
        text=ct['ChurnRate'].apply(lambda x: f'{x:.1f}%'),
        textposition='outside',
        showlegend=False
    ),
    row=1,
    col=2
)

# Axis styling
fig.update_yaxes(
    title_text='Churn Rate (%)',
    color='white',
    row=1,
    col=2
)

# Dark Theme
fig.update_layout(
    height=420,

    title={
        'text': '🍩 Churn Overview Dashboard',
        'x': 0.5,
        'xanchor': 'center'
    },

    title_font_size=18,
    title_font_color='white',

    paper_bgcolor='#1E293B',
    plot_bgcolor='#1E293B',

    font=dict(
        color='white',
        size=12
    ),

    margin=dict(
        l=40,
        r=40,
        t=80,
        b=40
    )
)

# Subplot title color
for annotation in fig['layout']['annotations']:
    annotation['font'] = dict(
        size=14,
        color='white'
    )

fig.show()

## 📡 Service & Payment Analysis

In [4]:
fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=[
        '📡 Churn by Internet Service',
        '💳 Churn by Payment Method'
    ]
)

# Internet Service
is_ = df.groupby('InternetService')['Churn'].mean().mul(100).round(1).reset_index()
is_.columns = ['Service', 'ChurnRate']
is_ = is_.sort_values('ChurnRate', ascending=False)

fig.add_trace(
    go.Bar(
        x=is_['Service'],
        y=is_['ChurnRate'],
        marker_color=['#c0392b', '#2980b9', '#27ae60'],
        text=is_['ChurnRate'].apply(lambda x: f'{x:.1f}%'),
        textposition='outside',
        showlegend=False
    ),
    row=1,
    col=1
)

# Payment Method
pm = df.groupby('PaymentMethod')['Churn'].mean().mul(100).round(1).reset_index()
pm.columns = ['Method', 'ChurnRate']
pm = pm.sort_values('ChurnRate', ascending=False)

fig.add_trace(
    go.Bar(
        x=pm['Method'],
        y=pm['ChurnRate'],
        marker_color='#8e44ad',
        text=pm['ChurnRate'].apply(lambda x: f'{x:.1f}%'),
        textposition='outside',
        showlegend=False
    ),
    row=1,
    col=2
)

# Axis
fig.update_yaxes(
    title_text='Churn Rate (%)',
    color='white'
)

fig.update_xaxes(
    color='white'
)

# Layout
fig.update_layout(
    height=420,

    title={
        'text': '📡 Service & Payment Insights',
        'x': 0.5,
        'xanchor': 'center'
    },

    title_font_size=18,
    title_font_color='white',

    paper_bgcolor='#1E293B',
    plot_bgcolor='#1E293B',

    font=dict(
        color='white',
        size=12
    ),

    margin=dict(
        l=40,
        r=40,
        t=80,
        b=40
    )
)

# Subplot title color
for annotation in fig['layout']['annotations']:
    annotation['font'] = dict(
        size=14,
        color='white'
    )

fig.show()

## ⏱️ Tenure & Churn Reasons

In [5]:
fig = make_subplots(
    rows=1,
    cols=2,
    specs=[[{'type':'bar'}, {'type':'pie'}]],
    subplot_titles=[
        '⏱️ Churn Rate by Tenure Group',
        '🚪 Why Customers Leave'
    ]
)

# Tenure Group Chart
tg = df.groupby('Tenure_Group', observed=True)['Churn'].mean().mul(100).round(1).reset_index()
tg.columns = ['Group', 'ChurnRate']

fig.add_trace(
    go.Bar(
        x=tg['Group'],
        y=tg['ChurnRate'],
        marker_color=['#c0392b','#e67e22','#f1c40f','#2980b9','#27ae60'],
        text=tg['ChurnRate'].apply(lambda x: f'{x:.1f}%'),
        textposition='outside',
        showlegend=False
    ),
    row=1,
    col=1
)

# Churn Reasons Pie Chart
reasons = df[df['Churn']==1]['ChurnReason'].value_counts().reset_index()
reasons.columns = ['Reason', 'Count']

fig.add_trace(
    go.Pie(
        labels=reasons['Reason'],
        values=reasons['Count'],
        hole=0.45,
        textinfo='label+percent',
        marker_colors=px.colors.qualitative.Pastel
    ),
    row=1,
    col=2
)

# Axis
fig.update_yaxes(
    title_text='Churn Rate (%)',
    color='white',
    row=1,
    col=1
)

fig.update_xaxes(
    color='white',
    row=1,
    col=1
)

# Layout
fig.update_layout(
    height=450,

    title={
        'text': '⏱️ Tenure & Churn Reasons Analysis',
        'x': 0.5,
        'xanchor': 'center'
    },

    title_font_size=18,
    title_font_color='white',

    showlegend=False,

    paper_bgcolor='#1E293B',
    plot_bgcolor='#1E293B',

    font=dict(
        color='white',
        size=12
    ),

    margin=dict(
        l=40,
        r=40,
        t=80,
        b=40
    )
)

# Subplot Titles White
for annotation in fig['layout']['annotations']:
    annotation['font'] = dict(
        size=14,
        color='white'
    )

fig.show()

## 📅 Cohort Retention Trend

In [6]:
cohort = df.groupby('CohortMonth').agg(
    Total=('Churn', 'count'),
    Churned=('Churn', 'sum')
).reset_index()

cohort['RetentionRate'] = (
    (cohort['Total'] - cohort['Churned']) /
    cohort['Total'] * 100
).round(1)

cohort['ChurnRate'] = (
    cohort['Churned'] /
    cohort['Total'] * 100
).round(1)

cohort = cohort.sort_values('CohortMonth')

fig = go.Figure()

# Retention Line
fig.add_trace(
    go.Scatter(
        x=cohort['CohortMonth'],
        y=cohort['RetentionRate'],
        mode='lines+markers',
        name='🟢 Retention Rate %',
        line=dict(color='#27ae60', width=3),
        marker=dict(size=8, symbol='circle'),
        fill='tozeroy',
        fillcolor='rgba(39,174,96,0.15)'
    )
)

# Churn Line
fig.add_trace(
    go.Scatter(
        x=cohort['CohortMonth'],
        y=cohort['ChurnRate'],
        mode='lines+markers',
        name='🔴 Churn Rate %',
        line=dict(color='#c0392b', width=3, dash='dot'),
        marker=dict(size=8, symbol='diamond')
    )
)

# Average Retention Line
fig.add_hline(
    y=cohort['RetentionRate'].mean(),
    line_dash='dash',
    line_color='#27ae60',
    annotation_text=f'Avg Retention: {cohort["RetentionRate"].mean():.1f}%'
)

# Layout
fig.update_layout(
    height=420,

    title={
        'text': '📅 Monthly Cohort Retention Trend',
        'x': 0.5,
        'xanchor': 'center'
    },

    title_font_size=18,
    title_font_color='white',

    xaxis_title='📆 Join Month',
    yaxis_title='📊 Rate (%)',

    xaxis_tickangle=-45,

    legend=dict(
        orientation='h',
        y=1.12,
        font=dict(color='white')
    ),

    paper_bgcolor='#1E293B',
    plot_bgcolor='#1E293B',

    font=dict(
        color='white',
        size=12
    ),

    margin=dict(
        l=40,
        r=40,
        t=80,
        b=40
    )
)

# Axis Colors
fig.update_xaxes(
    color='white',
    showgrid=True,
    gridcolor='rgba(255,255,255,0.15)'
)

fig.update_yaxes(
    color='white',
    showgrid=True,
    gridcolor='rgba(255,255,255,0.15)'
)

fig.show()

## 🔵 Tenure vs Monthly Charges

In [9]:
sample = df.sample(700, random_state=99)

fig = px.scatter(
    sample,
    x='Tenure_Months',
    y='MonthlyCharges',
    color='Churn_Label',
    color_discrete_map=COLORS,
    hover_data=['CustomerID', 'Contract', 'InternetService'],
    opacity=0.65,
    size_max=8,

    title='🔵 Tenure vs Monthly Charges by Churn Status',

    labels={
        'Tenure_Months': '⏱️ Tenure (Months)',
        'MonthlyCharges': '💰 Monthly Charges (₹)',
        'Churn_Label': '📊 Status'
    }
)

fig.update_traces(
    marker=dict(
        size=7,
        line=dict(width=1, color='white')
    )
)

fig.update_layout(
    height=450,

    title={
        'text': '🔵 Tenure vs Monthly Charges by Churn Status',
        'x': 0.5,
        'xanchor': 'center'
    },

    title_font_size=18,
    title_font_color='white',

    paper_bgcolor='#1E293B',
    plot_bgcolor='#1E293B',

    font=dict(
        color='white',
        size=12
    ),

    legend=dict(
        title='📊 Customer Status',
        font=dict(color='white')
    ),

    margin=dict(
        l=40,
        r=40,
        t=80,
        b=40
    )
)

# Axis Styling
fig.update_xaxes(
    color='white',
    showgrid=True,
    gridcolor='rgba(255,255,255,0.15)'
)

fig.update_yaxes(
    color='white',
    showgrid=True,
    gridcolor='rgba(255,255,255,0.15)'
)

fig.show()

## 💡 Business Recommendations

| Priority | Recommendation |
|----------|---------------|
| 🔴 High | Offer discounts to convert Month-to-Month → Annual plans |
| 🔴 High | 90-day onboarding program for new customers |
| 🟡 Medium | Improve Fiber Optic service quality |
| 🟡 Medium | Incentivize auto-pay adoption |
| 🟢 Low | Cross-sell bundles to single-product customers |
| 🟢 Low | Loyalty rewards at 24-month milestone |